#
RLCT Estimation of Sorting

This Jupyter Notebook aims to measure the Real Log Canonical Threshold (RLCT) for a small 3-layer transformer model (~280,000 parameters) trained to sort sequences of 20 digits consisting of the numbers 0-19. It uses both Stochastic Gradient Nose-Hoover Thermostat (SGNHT) and Stochastic Gradient Langevin Dynamics (SGLD) as sampling methods.

## Main Steps:

1. **Data Preparation**: Generate the dataset of numbers to sort.
2. **Model Training**: Train a transformer model using stochastic gradient descent.
3. **Model Evaluation**: Evaluate the model's performance on a test set.
4. **RLCT Estimation**: Use SGNHT and SGLD samplers to estimate RLCT.
5. **Plotting**: Visualize train and test losses, and RLCT estimates.

In [1]:
%pip install devinterp seaborn torchvision pickleshare wandb plotly einops scikit-learn
!git clone https://github.com/ucla-vision/entropy-sgd.git
%cd entropy-sgd
from python.optim import EntropySGD
%cd ..

Defaulting to user installation because normal site-packages is not writeable
  Using cached matplotlib-3.10.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (8.6 MB)
  Using cached cloudpickle-3.1.1-py3-none-any.whl (20 kB)

[notice] A new release of pip is available: 23.1.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
fatal: destination path 'entropy-sgd' already exists and is not an empty directory.
/gpfs/home1/bshaffrey/entropy-sgd
/gpfs/home1/bshaffrey


In [2]:
import numpy as np
import torch as t
import torch
import torch.nn as nn
import torch.optim as optim
import time
import torch.nn.functional as F
import einops
import random
import helpers
from transformers import *
from dataclasses import dataclass
import os
import copy
import wandb
from tqdm.notebook import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from python.optim import EntropySGD
from torch.utils.data import DataLoader

from devinterp.optim.sgld import SGLD
from devinterp.optim.sgnht import SGNHT

PRIMARY, SECONDARY, TERTIARY, QUATERNARY = sns.color_palette("muted")[:4]
PRIMARY_LIGHT, SECONDARY_LIGHT, TERTIARY_LIGHT, QUATERNARY_LIGHT = sns.color_palette(
    "pastel"
)[:4]


In [3]:
def full_accuracy(config, model, data):
    logits = model(data)[:, -1]
    labels = t.tensor([config.fn(i, j) for i, j, _ in data]).to(config.device)
    return (logits.argmax(1) == labels).float().mean()

def accuracy_function(outputs, targets):
    return (outputs[ : , -1].argmax(1) == targets).float().mean()

def do_a_training_step(config, model, train_data, test_data, optimizer, scheduler, epoch: int):
        '''returns train_loss, test_loss'''
        model.train()
        train_loss = full_loss(config=config, model=model, data=train_data)
        train_accuracy = full_accuracy(config=config, model=model, data=train_data)
        #self.train_losses.append(train_loss.item())
        #self.test_losses.append(test_loss.item())
        train_loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        model.eval()  # Set model to evaluation mode
        with torch.no_grad():  # Disable gradient calculation for test
            test_loss = full_loss(config=config, model=model, data=test_data)
            test_accuracy = full_accuracy(config=config, model=model, data=test_data)
            
        if epoch % 100 == 0:
            # TODO is this ok? this was np.log, and it was barking at me ; i think np.log was being interpreted as a logging module
            print(f'Epoch {epoch}, train loss {t.log(train_loss).item():.4f}, test loss {t.log(test_loss).item():.4f}')

        return train_loss.detach(), test_loss.detach(), train_accuracy.detach(), test_accuracy.detach()

def train_one_epoch(model, train_loader, optimizer, scheduler, criterion, model_key):

    model.train()
    train_loss = 0
    train_accuracy = 0
    for index, (data, targets) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(data.to(DEVICE))
        loss = criterion(outputs, targets.to(DEVICE))
        train_loss += loss.detach().item()
        train_accuracy += accuracy_function(outputs, targets.to(DEVICE))
        loss.backward()
        optimizer.step()
        scheduler.step()
        yield (train_loss, train_accuracy)


def evaluate(model, test_loader, criterion):
    model.eval()
    test_loss = 0
    test_accuracy = 0
    with torch.no_grad():
        for index, (data, targets) in enumerate(test_loader):
            outputs = model(data.to(DEVICE))
            loss = criterion(outputs, targets.to(DEVICE))
            test_loss += loss.item()
            test_accuracy += accuracy_function(outputs, targets.to(DEVICE))

    yield (test_loss, test_accuracy)


In [4]:
# Constants
DEVICE = "cuda" if t.cuda.is_available() else "cpu"
BATCH_SIZE = 16384
LR = 1e-4
N_EPOCHS = 30000
SAVE_EVERY_N_EPOCHS = 100
config = Config()

def get_data(config : Config):
    num_to_generate = config.p
    pairs = [(i, j, num_to_generate) for i in range(num_to_generate) for j in range(num_to_generate)]
    random.seed(config.seed)
    random.shuffle(pairs)
    div = int(config.frac_train*len(pairs))
    labels = [config.fn(i, j) for i, j, _ in pairs]
    pairs = t.tensor(pairs).long()
    labels = t.tensor(labels).long()
    train_data = list(zip(pairs[:div], labels[:div]))
    test_data = list(zip(pairs[div:], labels[div:]))
    return train_data, test_data, pairs, labels

train_data, test_data, all_data, labels = get_data(config = config)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
criterion = helpers.cross_entropy_high_precision
print(len(train_loader))
print(len(test_loader))
print(len(train_data))
print(len(test_data))
print(torch.cuda.is_available())
print(config.device)

1
1
3830
8939
True
cuda


In [ ]:
def train_models(config, train_data, test_data, all_data, labels, runs):
    save_every_n_epochs = SAVE_EVERY_N_EPOCHS
    num_save_steps = N_EPOCHS // save_every_n_epochs
    train_losses = torch.zeros(runs, num_save_steps)
    test_losses = torch.zeros(runs, num_save_steps)
    trig_losses = torch.zeros(runs, num_save_steps)
    excluded_losses = torch.zeros(runs, num_save_steps)
    train_accuracies = torch.zeros(runs, num_save_steps)
    test_accuracies = torch.zeros(runs, num_save_steps)
    models_saved = []
    
    fourier_basis = make_fourier_basis(config = config)
    train, _ = gen_train_test(config = config)
    is_train, is_test = config.is_train_is_test(train = train)
    all_data = torch.tensor([(i, j, config.p) for i in range(config.p) for j in range(config.p)]).to(config.device)
    labels = torch.tensor([config.fn(i, j) for i, j, _ in all_data]).to(config.device)
    indices = get_indices_for_key_freqs_calculation(config)
    
    for run in tqdm(range(runs)):
        model = Transformer(config, use_cache=False)
        model.to(config.device)
        print(sum(p.numel() for p in model.parameters()))
        optimizer = optim.AdamW(model.parameters(), lr = config.lr, weight_decay=config.weight_decay, betas=(0.9, 0.98))
        scheduler = optim.lr_scheduler.LambdaLR(optimizer, lambda step: min(step/10, 1)) # TODO make this a config option
        index = 0
        for epoch in tqdm(range(N_EPOCHS)):
            torch.cuda.empty_cache()
            (train_loss, train_accuracy) = next(train_one_epoch(
                model, train_loader, optimizer, scheduler, criterion, 'sgd'
            ))
            (test_loss, test_accuracy) = next(evaluate(model, test_loader, criterion))
            if epoch % 100 == 0:
              print(
                  f"Epoch {epoch+1}, Model {'sgd'.upper()} Train Loss: {train_loss}, Test Loss: {test_loss}", '\n',
                  f"Epoch {epoch+1}, Model {'sgd'.upper()} Train Accuracy: {train_accuracy}, Test Accuracy: {test_accuracy}"
              )
            #if (epoch % 10 == 0 and epoch < 1000) or epoch % 100 == 0:
            if epoch % save_every_n_epochs == 0:
                train_losses[run, index] = train_loss
                test_losses[run, index] = test_loss
                train_accuracies[run, index] = train_accuracy
                test_accuracies[run, index] = test_accuracy
                models_saved += [copy.deepcopy(model)]
                
                with torch.no_grad():  
                    
                    key_freqs = calculate_key_freqs_vectorised(config = config, model = model, all_data = all_data, labels=labels, fourier_basis=fourier_basis, indices=indices) 
                    #key_freqs = calculate_key_freqs(config = config, model = model, all_data = all_data, labels=labels, fourier_basis=fourier_basis)
                    '''
                    if epoch >= 29000:
                        key_freqs_old = calculate_key_freqs(config = config, model = model, all_data = all_data, labels=labels, fourier_basis=fourier_basis)
                        print(key_freqs)
                        print(key_freqs_old)
                    '''
                    
                    logits = model(all_data)[:, -1, :-1] # TODO i think this is equivalent to what's in the new paper?
                
                    trig_losses[run, index], trig_logits = calculate_trig_loss(config = config,
                    model = model,
                    train = train,
                    logits = logits,
                    key_freqs = key_freqs,
                    fourier_basis=fourier_basis,
                    all_data=all_data,
                    is_test=is_test,
                    is_train=is_train,
                    labels=labels)
                
                    excluded_losses[run, index] = calculate_excluded_loss(
                    config=config,
                    is_train=is_train,
                    is_test=is_test,
                    labels=labels,
                    logits=logits,
                    trig_logits=trig_logits)
                
                index += 1

    train_losses_final = train_losses.mean(dim=0)
    test_losses_final = test_losses.mean(dim=0)
    train_accuracies_final = train_accuracies.mean(dim=0)
    test_accuracies_final = test_accuracies.mean(dim=0)
    trig_losses_final = trig_losses.mean(dim=0)
    excluded_losses_final = excluded_losses.mean(dim=0)
    torch.cuda.empty_cache()

    return train_losses_final, test_losses_final, train_accuracies_final, test_accuracies_final, trig_losses_final, excluded_losses_final, models_saved

def train_models_orig(config, runs):
    train, test = gen_train_test(config = config)
    train_losses = torch.zeros(runs, N_EPOCHS)
    test_losses = torch.zeros(runs, N_EPOCHS)
    train_accuracies = torch.zeros(runs, N_EPOCHS)
    test_accuracies = torch.zeros(runs, N_EPOCHS)
    models_saved = []
    for run in tqdm(range(runs)):
        model = Transformer(config, use_cache=False)
        model.to(config.device)
        optimizer = optim.AdamW(model.parameters(), lr = config.lr, weight_decay=config.weight_decay, betas=(0.9, 0.98))
        scheduler = optim.lr_scheduler.LambdaLR(optimizer, lambda step: min(step/10, 1)) # TODO make this a config option
        for epoch in tqdm(range(N_EPOCHS)):
            train_loss, test_loss, train_accuracy, test_accuracy = do_a_training_step(config, model, train, test, optimizer, scheduler, epoch)
            train_losses[run, epoch] = train_loss
            test_losses[run, epoch] = test_loss
            train_accuracies[run, epoch] = train_accuracy
            test_accuracies[run, epoch] = test_accuracy
            #models_saved += [copy.deepcopy(model)]
            if epoch % 100 == 0:
              print(
                  f"Epoch {epoch+1}, Model {'sgd'.upper()} Train Loss: {train_loss}, Test Loss: {test_loss}", '\n',
                  f"Epoch {epoch+1}, Model {'sgd'.upper()} Train Accuracy: {train_accuracy}, Test Accuracy: {test_accuracy}"
              )
    train_losses_final = train_losses.mean(dim=0)
    test_losses_final = test_losses.mean(dim=0)
    train_accuracies_final = train_accuracies.mean(dim=0)
    test_accuracies_final = test_accuracies.mean(dim=0)
    torch.cuda.empty_cache()
    
    return train_losses_final, test_losses_final, train_accuracies_final, test_accuracies_final, models_saved

torch.cuda.empty_cache()
runs = 1
train_losses_final, test_losses_final, train_accuracies_final, test_accuracies_final, trig_losses_final, excluded_losses_final, models_saved = train_models(config, train_loader, test_loader, all_data, labels, runs)
#train_losses_final, test_losses_final, train_accuracies_final, test_accuracies_final, models_saved = train_models_orig(config, runs)

Epoch 10001, Model SGD Train Loss: 2.702470350430254e-07, Test Loss: 13.83983934778715 
 Epoch 10001, Model SGD Train Accuracy: 1.0, Test Accuracy: 0.1782078593969345
Epoch 10101, Model SGD Train Loss: 2.6872452374171407e-07, Test Loss: 13.470596448079657 
 Epoch 10101, Model SGD Train Accuracy: 1.0, Test Accuracy: 0.18458440899848938
Epoch 10201, Model SGD Train Loss: 2.676715524756826e-07, Test Loss: 13.09823858956962 
 Epoch 10201, Model SGD Train Accuracy: 1.0, Test Accuracy: 0.19017787277698517
Epoch 10301, Model SGD Train Loss: 2.6628153894135715e-07, Test Loss: 12.714917137565655 
 Epoch 10301, Model SGD Train Accuracy: 1.0, Test Accuracy: 0.198008731007576
Epoch 10401, Model SGD Train Loss: 2.645394280316626e-07, Test Loss: 12.321819258974847 
 Epoch 10401, Model SGD Train Accuracy: 1.0, Test Accuracy: 0.20427341759204865
Epoch 10501, Model SGD Train Loss: 2.6326300132343214e-07, Test Loss: 11.924173256648814 
 Epoch 10501, Model SGD Train Accuracy: 1.0, Test Accuracy: 0.210649

Epoch 14901, Model SGD Train Loss: 1.1478370162828529e-07, Test Loss: 5.70645968536785e-05 
 Epoch 14901, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 15001, Model SGD Train Loss: 1.1267404210781815e-07, Test Loss: 2.0789636136381127e-05 
 Epoch 15001, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 15101, Model SGD Train Loss: 1.1094736020215368e-07, Test Loss: 8.657067485218918e-06 
 Epoch 15101, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 15201, Model SGD Train Loss: 1.0953032715183114e-07, Test Loss: 4.253156111847128e-06 
 Epoch 15201, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 15301, Model SGD Train Loss: 1.0828252202039058e-07, Test Loss: 2.368416094693859e-06 
 Epoch 15301, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 15401, Model SGD Train Loss: 1.0719169304021963e-07, Test Loss: 1.5376104502728684e-06 
 Epoch 15401, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 15501, Model SGD Train Loss: 1.0624810398870664e-07

Epoch 20201, Model SGD Train Loss: 9.855773315794235e-08, Test Loss: 4.063600078032527e-07 
 Epoch 20201, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 20301, Model SGD Train Loss: 9.853050319974434e-08, Test Loss: 4.1238132662371974e-07 
 Epoch 20301, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 20401, Model SGD Train Loss: 9.850700727359458e-08, Test Loss: 4.1894954389603075e-07 
 Epoch 20401, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 20501, Model SGD Train Loss: 9.848173494630247e-08, Test Loss: 4.268471528398813e-07 
 Epoch 20501, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 20601, Model SGD Train Loss: 9.845801708264215e-08, Test Loss: 4.352145150716085e-07 
 Epoch 20601, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 20701, Model SGD Train Loss: 9.843450688799653e-08, Test Loss: 4.421287962300094e-07 
 Epoch 20701, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 20801, Model SGD Train Loss: 9.841063484462265e-08, Test

Epoch 25501, Model SGD Train Loss: 9.773934211893339e-08, Test Loss: 1.2399719338328095e-06 
 Epoch 25501, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 25601, Model SGD Train Loss: 9.772857225229576e-08, Test Loss: 1.2730316075082845e-06 
 Epoch 25601, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 25701, Model SGD Train Loss: 9.772225883453305e-08, Test Loss: 1.290470233216977e-06 
 Epoch 25701, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 25801, Model SGD Train Loss: 9.771037684588112e-08, Test Loss: 1.299286221339505e-06 
 Epoch 25801, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 25901, Model SGD Train Loss: 9.770037994849813e-08, Test Loss: 1.315700154029787e-06 
 Epoch 25901, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 26001, Model SGD Train Loss: 9.769037458671015e-08, Test Loss: 1.3299732189259686e-06 
 Epoch 26001, Model SGD Train Accuracy: 1.0, Test Accuracy: 1.0
Epoch 26101, Model SGD Train Loss: 9.768309313863049e-08, Tes

In [5]:
from devinterp.slt import estimate_learning_coeff_with_summary

def estimate_rlcts(models, train_loader, criterion, data_length, device, num_draws, num_models):
    estimates = {"sgnht": [], "sgld": []}
    for idx, model in enumerate(tqdm(models)):
        for method, optimizer_kwargs in [
            #("sgnht", {"lr": 1e-7, "diffusion_factor": 0.01}),
            ("sgld", {"lr": 1e-5, "localization": 100.0, "noise_level": 1.0}),
        ]:
            results = estimate_learning_coeff_with_summary(
                model,
                train_loader,
                criterion=criterion,
                optimizer_kwargs=optimizer_kwargs,
                sampling_method=SGNHT if method == "sgnht" else SGLD,
                num_chains=1,
                num_draws=num_draws,
                num_burnin_steps=100,
                num_steps_bw_draws=1,
                device=device,
                seed=0
            )
            estimate = results["llc/mean"]
            estimates[method].append(estimate)
    return estimates

def obtain_rlct_estimates(train_loader, models_saved, criterion, runs):
    #num_models = 100 + (N_EPOCHS - 1000) // 100 if N_EPOCHS > 1000 else N_EPOCHS // 10
    num_models = N_EPOCHS // SAVE_EVERY_N_EPOCHS
    data_length = len(train_loader)
    rlct_estimates = {"sgnht": torch.zeros(runs, num_models), "sgld": torch.zeros(runs, num_models)}
    num_draws = 400

    for run in tqdm(range(runs)):
        rlct_estimate = estimate_rlcts(
            models_saved[num_models * run : num_models * (run + 1)], train_loader, criterion, data_length, DEVICE, num_draws, num_models
        )
        #rlct_estimates["sgnht"][run] = torch.tensor(rlct_estimate["sgnht"])
        rlct_estimates["sgld"][run] = torch.tensor(rlct_estimate["sgld"])

    rlct_estimates_final = {"sgnht": rlct_estimates["sgnht"].mean(dim=0), "sgld": rlct_estimates["sgld"].mean(dim=0)}
    return rlct_estimates_final

#rlct_estimates_final = obtain_rlct_estimates(train_loader, models_saved, criterion, runs)

ImportError: cannot import name 'estimate_learning_coeff_with_summary' from 'devinterp.slt' (/home/bshaffrey/.local/lib/python3.11/site-packages/devinterp/slt/__init__.py)

In [8]:
dataset = 25

def plot_losses(train_losses_final, test_losses_final, other_loss, name_other_loss, dataset):

    sns.set_style("whitegrid")
    # more frequent checkpoints within first 1000 epochs
    #first_part = np.arange(1, 1001, 10)
    # less frequent thereafter
    #second_part = np.arange(1001, N_EPOCHS, 100)
    # Combine the two arrays
    #x_axis = np.concatenate([first_part, second_part])
    x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss", color=PRIMARY)
    plt.yscale('log')
    ax1.plot(x_axis, train_losses_final, label="Train Loss, sgd", color=PRIMARY)
    ax1.plot(x_axis, test_losses_final, label="Test Loss, sgd", color=PRIMARY_LIGHT)
    ax1.plot(x_axis, other_loss.detach(), label=name_other_loss, color=SECONDARY_LIGHT)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("losses_" + str(dataset) + "_" + name_other_loss + "_" + str(N_EPOCHS) + "_epochs.png")

def plot_accuracies(train_accuracies_final, test_accuracies_final, dataset):

    sns.set_style("whitegrid")
    
    #first_part = np.arange(1, 1001, 10)
    
    # Create array from 1000 to 50000 with step 100
    # Start from 1100 to avoid duplicating 1000
    #second_part = np.arange(1001, N_EPOCHS, 100)
    
    # Combine the two arrays
    #x_axis = np.concatenate([first_part, second_part])
    x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy", color=PRIMARY)
    plt.yscale('log')
    ax1.plot(x_axis, train_accuracies_final, label="Train Accuracy, sgd", color=PRIMARY)
    ax1.plot(x_axis, test_accuracies_final, label="Test Accuracy, sgd", color=PRIMARY_LIGHT)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("accuracies_" + str(dataset) + "_" + str(N_EPOCHS) + "_epochs.png")

def plot_rlcts(rlct_estimates_final, dataset, rlct_estimates_final_other = {}):

    sns.set_style("whitegrid")
    
    #first_part = np.arange(1, 1001, 10)
    
    # Create array from 1000 to 50000 with step 100
    # Start from 1100 to avoid duplicating 1000
    #second_part = np.arange(1001, N_EPOCHS, 100)
    
    # Combine the two arrays
    #x_axis = np.concatenate([first_part, second_part])
    x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)

    fig, ax2 = plt.subplots(figsize=(10, 6))
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel(r"Local Learning Coefficient, $\hat \lambda$", color=SECONDARY)
    if rlct_estimates_final_other:
        ax2.plot(x_axis, rlct_estimates_final_other["sgld"], label="summed curve", color=TERTIARY)
    ax2.plot(x_axis, rlct_estimates_final["sgld"], label="SGLD, sgd", color=TERTIARY_LIGHT)
    ax2.axvline(x=1400, color='r', linestyle='--', linewidth=1)
    ax2.axvline(x=9400, color='r', linestyle='--', linewidth=1)
    ax2.axvline(x=14000, color='r', linestyle='--', linewidth=1)
    ax2.tick_params(axis="y", labelcolor=SECONDARY)
    ax2.legend(loc="center right")

    fig.tight_layout()
    plt.show()
    fig.savefig("rclt_" + dataset + "_" + str(N_EPOCHS) + "_epochs.png")
    
def plot_rlcts_circuits(rlct_estimates_final_gen, rlct_estimates_final_mem, rlct_estimates_final_other):

    sns.set_style("whitegrid")
    
    x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)

    fig, ax2 = plt.subplots(figsize=(10, 6))
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel(r"Local Learning Coefficient, $\hat \lambda$", color=SECONDARY)
    ax2.plot(x_axis, rlct_estimates_final_gen["sgld"], label="Gen", color=PRIMARY_LIGHT)
    ax2.plot(x_axis, rlct_estimates_final_mem["sgld"], label="Mem", color=TERTIARY)
    ax2.plot(x_axis, rlct_estimates_final_other["sgld"], label="Other", color=QUATERNARY)
    ax2.axvline(x=1400, color='r', linestyle='--', linewidth=1)
    ax2.axvline(x=9400, color='r', linestyle='--', linewidth=1)
    ax2.axvline(x=14000, color='r', linestyle='--', linewidth=1)
    ax2.tick_params(axis="y", labelcolor=SECONDARY)
    ax2.legend(loc="center right")

    fig.tight_layout()
    plt.show()
    fig.savefig("rclt_circuits_" + str(N_EPOCHS) + "_epochs.png")

def plot_losses_chain(last_chain_losses_final, dataset):
    sns.set_style("whitegrid")

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Draw")
    ax1.set_ylabel("Loss", color=PRIMARY)
    ax1.plot(last_chain_losses_final, label="Loss, sgd", color=PRIMARY)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("last_chain_losses_" + str(dataset) + "_" + str(N_EPOCHS) + "_epochs.png")

plot_losses(train_losses_final, test_losses_final, trig_losses_final, 'Restricted_Loss', dataset)
plot_losses(train_losses_final, test_losses_final, excluded_losses_final, 'Excluded_Loss', dataset)
plot_accuracies(train_accuracies_final, test_accuracies_final, dataset)
plot_rlcts(rlct_estimates_final, dataset='full')
#plot_losses_chain(last_chain_losses_final, dataset)

/scratch-local/bshaffrey.7918365/ipykernel_2176334/3398407666.py:24: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/scratch-local/bshaffrey.7918365/ipykernel_2176334/3398407666.py:50: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/scratch-local/bshaffrey.7918365/ipykernel_2176334/3398407666.py:80: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [25]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

num_models = N_EPOCHS // SAVE_EVERY_N_EPOCHS

def d_dt(steps, values):
    slope = np.zeros(len(steps))

    # Compute Slope and Curvature
    for i in range(1, len(steps) - 1):
        dx1 = steps[i+1] - steps[i]
        dx0 = steps[i] - steps[i-1]
        
        dy1 = values[i+1] - values[i]
        dy0 = values[i] - values[i-1]
        
        slope[i] = (dy1 / dx1 + dy0 / dx0) / 2

    slope[0] = slope[1]
    slope[-1] = slope[-2]

    return slope
'''
first_part = np.arange(1, 1001, 10)
    
# Create array from 1000 to 50000 with step 100
# Start from 1100 to avoid duplicating 1000
second_part = np.arange(1001, N_EPOCHS, 100)
    
# Combine the two arrays
x_axis = np.concatenate([first_part, second_part])
'''
x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)
_steps = x_axis.reshape((-1, 1))
_y = rlct_estimates_final['sgld']

kernel = C(1.0, (1e-3, 1e3)) * RBF(3, (5e-1, 1e3))

# Create a Gaussian Process Regressor
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)

# Fit the Gaussian Process
gp.fit(_steps, _y)
_ypred = gp.predict(_steps)
_derivy = d_dt(_steps, _ypred)
print(x_axis[np.abs(_derivy) < .25])


sns.set_style("whitegrid")

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.set_xlabel("Epoch")
ax1.set_ylabel("d lambda / dt", color=PRIMARY)
#plt.yscale('log')
ax1.plot(x_axis, _derivy, label="Slope, sgd", color=PRIMARY_LIGHT)
plt.axhline(y=0, color='k', linestyle=':', linewidth=1)
plt.axvline(x=1400, color='r', linestyle='--', linewidth=1)
plt.axvline(x=9400, color='r', linestyle='--', linewidth=1)
plt.axvline(x=14000, color='r', linestyle='--', linewidth=1)
ax1.tick_params(axis="y", labelcolor=PRIMARY)
ax1.legend(loc="upper left")
fig.tight_layout()
plt.show()
fig.savefig("slope_rlct_" + str(dataset) + "_" + str(N_EPOCHS) + "_epochs.png")

[ 1401  1501  1601  1701  1801  1901  2001  2101  2201  2301  2401  2501
  2601  2701  2801  2901  3001  3101  3201  3301  3401  3501  3601  3701
  3801  3901  4001  4101  4201  4301  4401  4501  4601  5401  5501  5601
  5701  5801  5901  6001  6101  6201  6301  6401  6501  6601  6701  6801
  6901  7001  7101  7201  7301  7401  7501  7601  7701  7801  7901  8001
  8101  8201  8301  8401  8501  8601  8701  8801  8901  9001  9101  9201
  9301  9401  9501  9601  9701  9801  9901 10001 10101 10201 10301 10401
 10501 10601 10701 10801 10901 11001 11101 11201 11301 11401 11501 11601
 11701 11801 11901 12001 12101 12201 12301 12401 12501 12601 12701 12801
 12901 13001 13101 13201 13301 13401 13501 13601 13701 13801 13901 14001
 14101 14201 14301 14401 14501 14601 14701 14801 14901 15001 15101 15201
 15301 15401 15501 15601 15701 15801 15901 16001 16101 16201 16301 16401
 16501 16601 16701 16801 16901 17001 17101 17201 17301 17401 17501 17601
 17701 17801 17901 18001 18101 18201 18301 18401 18

/home/bshaffrey/.local/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/scratch-local/bshaffrey.7546692/ipykernel_2629260/2847878110.py:63: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [12]:
torch.save(models_saved, 'models_saved_modular_addition.pt')
torch.save(train_losses_final, 'train_losses_final_modular_addition.pt')
torch.save(test_losses_final, 'test_losses_final_modular_addition.pt')
torch.save(train_accuracies_final, 'train_accuracies_final_modular_addition.pt')
torch.save(test_accuracies_final, 'test_accuracies_final_modular_addition.pt')
torch.save(rlct_estimates_final, 'rlct_estimates_final_modular_addition.pt')
torch.save(trig_losses_final, 'trig_losses_final_modular_addition.pt')
torch.save(excluded_losses_final, 'excluded_losses_final_modular_addition.pt')
torch.save(_derivy, 'derivy_modular_addition.pt')

In [11]:
models_saved = torch.load('models_saved_modular_addition.pt')
train_losses_final = torch.load('train_losses_final_modular_addition.pt')
test_losses_final = torch.load('test_losses_final_modular_addition.pt')
train_accuracies_final = torch.load('train_accuracies_final_modular_addition.pt')
test_accuracies_final = torch.load('test_accuracies_final_modular_addition.pt')
rlct_estimates_final = torch.load('rlct_estimates_final_modular_addition.pt')
trig_losses_final = torch.load('trig_losses_final_modular_addition.pt')
excluded_losses_final = torch.load('excluded_losses_final_modular_addition.pt')
_derivy = torch.load('derivy_modular_addition.pt')
trig_loss_diffs = torch.load('trig_loss_diffs_modular_addition.pt')
excluded_loss_diffs = torch.load('excluded_loss_diffs_modular_addition.pt')

In [6]:
rlct_estimates_final = torch.load('rlct_estimates_final.pt')

/scratch-local/bshaffrey.7918365/ipykernel_2176334/498774048.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  rlct_estimates_final = torch.load('rlct_estimates_final.pt')

In [26]:
def random_partition_of_weights(tensor):
    tensor_shape = tensor.shape

    num_elements = tensor.numel()

    # Step 2: Generate shuffled indices
    indices = torch.randperm(num_elements)

    # Step 3: Split the indices into three disjoint sets
    split_1 = num_elements // 3
    split_2 = 2 * split_1

    indices_A = indices[:split_1]
    indices_B = indices[split_1:split_2]
    indices_C = indices[split_2:]

    # Step 4: Create three masks
    mask_A = torch.zeros(num_elements, dtype=torch.bool)
    mask_B = torch.zeros(num_elements, dtype=torch.bool)
    mask_C = torch.zeros(num_elements, dtype=torch.bool)

    mask_A[indices_A] = True
    mask_B[indices_B] = True
    mask_C[indices_C] = True

    # Step 5: Reshape the masks to the original tensor shape
    mask_A = mask_A.view(tensor_shape)
    mask_B = mask_B.view(tensor_shape)
    mask_C = mask_C.view(tensor_shape)
    
    return mask_A, mask_B, mask_C

def return_topk_percent_mask(tensor, proportion):
    # Step 1: Flatten the tensor
    flattened_tensor = tensor.flatten()

    # Step 2: Determine K, where K is 20% of the total number of elements
    total_elements = flattened_tensor.numel()
    K = int(proportion * total_elements)

    # Step 3: Find the value of the K-th largest element
    topk_values, _ = torch.topk(flattened_tensor, K)
    threshold_value = topk_values[-1]

    # Step 4: Create a boolean mask of the top K values
    return tensor >= threshold_value


def unravel_index(index, shape):
    out = []
    for dim in reversed(shape):
        out.append(index % dim)
        index = index // dim
    return tuple(reversed(out))

def ablation_study(model, loss_fn, config):
    
    trig_loss_diffs = {}
    excluded_loss_diffs = {}
    
    for name, param in model.named_parameters():
        trig_loss_diffs[name] = torch.zeros(param.shape)
        excluded_loss_diffs[name] = torch.zeros(param.shape)

    print(' ablation study begins: ')
    fourier_basis = make_fourier_basis(config = config)
    train, _ = gen_train_test(config = config)
    is_train, is_test = config.is_train_is_test(train = train)
    all_data = torch.tensor([(i, j, config.p) for i in range(config.p) for j in range(config.p)]).to(config.device)
    labels = torch.tensor([config.fn(i, j) for i, j, _ in all_data]).to(config.device)
                
    indices = get_indices_for_key_freqs_calculation(config)
    key_freqs = calculate_key_freqs_vectorised(config = config, model = model, all_data = all_data, labels=labels, fourier_basis=fourier_basis, indices=indices) 
    logits = model(all_data)[:, -1, :-1] # TODO i think this is equivalent to what's in the new paper?
                
    trig_loss_baseline, trig_logits = calculate_trig_loss(config = config,
                    model = model,
                    train = train,
                    logits = logits,
                    key_freqs = key_freqs,
                    fourier_basis=fourier_basis,
                    all_data=all_data,
                    is_test=is_test,
                    is_train=is_train,
                    labels=labels)
                
    excluded_loss_baseline = calculate_excluded_loss(
                    config=config,
                    is_train=is_train,
                    is_test=is_test,
                    labels=labels,
                    logits=logits,
                    trig_logits=trig_logits)
    
    for name, param in model.named_parameters():
        param_shape = param.shape
        for idx in tqdm(range(param.numel()), desc=f'Ablating {name}'):
            with torch.no_grad():
                # Convert flat index i to multi-dimensional index for the original shape
                multi_idx = unravel_index(idx, param_shape)
                
                # Save the original weight value
                original_value = param[multi_idx].item()
                
                # Set the weight to zero
                param[multi_idx] = 0.0
                
                # Compute new metrics
                key_freqs = calculate_key_freqs_vectorised(config = config, model = model, all_data = all_data, labels=labels, fourier_basis=fourier_basis, indices=indices) 
                logits = model(all_data)[:, -1, :-1] # TODO i think this is equivalent to what's in the new paper?
                
                trig_loss, trig_logits = calculate_trig_loss(config = config,
                    model = model,
                    train = train,
                    logits = logits,
                    key_freqs = key_freqs,
                    fourier_basis=fourier_basis,
                    all_data=all_data,
                    is_test=is_test,
                    is_train=is_train,
                    labels=labels)
                
                excluded_loss = calculate_excluded_loss(
                    config=config,
                    is_train=is_train,
                    is_test=is_test,
                    labels=labels,
                    logits=logits,
                    trig_logits=trig_logits)
                
                # Calculate differences
                trig_loss_diff = trig_loss - trig_loss_baseline if trig_loss > trig_loss_baseline else trig_loss_baseline - trig_loss
                excluded_loss_diff = excluded_loss - excluded_loss_baseline if excluded_loss > excluded_loss_baseline else excluded_loss_baseline - excluded_loss
                
                trig_loss_diffs[name][multi_idx] = trig_loss_diff
                excluded_loss_diffs[name][multi_idx] = excluded_loss_diff
            
                # Restore the original weight
                param[multi_idx] = original_value
                torch.cuda.empty_cache()
    
    return trig_loss_diffs, excluded_loss_diffs

# Use the function
#trig_loss_diffs, excluded_loss_diffs = ablation_study(models_saved[-1], criterion, config)
#torch.save(trig_loss_diffs, 'trig_loss_diffs.pt')
#torch.save(excluded_loss_diffs, 'excluded_loss_diffs.pt')
trig_loss_diffs = torch.load('trig_loss_diffs.pt')
excluded_loss_diffs = torch.load('excluded_loss_diffs.pt')

gen_circuit_weights = 0
mem_circuit_weights = 0
overlap = 0

gen_model_indices = {}
mem_model_indices = {}
other_model_indices = {}

# Analyze results
for name in tqdm(trig_loss_diffs.keys()):
    print(f"Layer: {name}")
    '''
    proportion = 0.3
    trig_indices = return_topk_percent_mask(trig_loss_diffs[name], proportion)
    exc_indices = return_topk_percent_mask(excluded_loss_diffs[name], proportion + .2)
    both = trig_indices & exc_indices
    
    gen_model_indices[name] = trig_indices
    mem_model_indices[name] = exc_indices & ~both
    other_model_indices[name] = ~both
    
    above_eps_trig = trig_loss_diffs[name][gen_model_indices[name]]
    above_eps_excluded = excluded_loss_diffs[name][mem_model_indices[name]]
    present_in_both = trig_loss_diffs[name][both]
    
    #print('present in both: ', torch.count_nonzero(both).item())
    gen_circuit_weights += above_eps_trig.shape[0]
    mem_circuit_weights += above_eps_excluded.shape[0]
    overlap += present_in_both.shape[0]
    
    print((gen_model_indices[name] | mem_model_indices[name] | other_model_indices[name]).all())
    '''
    maskA, maskB, maskC = random_partition_of_weights(trig_loss_diffs[name])
    
    gen_model_indices[name] = maskA
    mem_model_indices[name] = maskB
    other_model_indices[name] = maskC
    
print('number of gen circuit weights: ', gen_circuit_weights)
print('number of mem circuit weights: ', mem_circuit_weights)
print('present in both: ', overlap)
print(sum(p.numel() for p in models_saved[-1].parameters()))

'''
topk_trig_loss_diffs, indices1 = torch.topk(np.abs(trig_loss_diffs), k)
topk_excluded_loss_diffs, indices2 = torch.topk(np.abs(excluded_loss_diffs), k)
A = weight_ids[indices1]
B = weight_ids[indices2]
print(np.intersect1d(A, B))
'''

  0%|          | 0/11 [00:00<?, ?it/s]

Layer: embed.W_E
Layer: pos_embed.W_pos
Layer: blocks.0.attn.W_K
Layer: blocks.0.attn.W_Q
Layer: blocks.0.attn.W_V
Layer: blocks.0.attn.W_O
Layer: blocks.0.mlp.W_in
Layer: blocks.0.mlp.b_in
Layer: blocks.0.mlp.W_out
Layer: blocks.0.mlp.b_out
Layer: unembed.W_U
number of gen circuit weights:  0
number of mem circuit weights:  0
present in both:  0
226816


'\ntopk_trig_loss_diffs, indices1 = torch.topk(np.abs(trig_loss_diffs), k)\ntopk_excluded_loss_diffs, indices2 = torch.topk(np.abs(excluded_loss_diffs), k)\nA = weight_ids[indices1]\nB = weight_ids[indices2]\nprint(np.intersect1d(A, B))\n'

In [83]:
k = 10000

print('number of gen circuit weights: ', gen_circuit_weights)
print('number of mem circuit weights: ', mem_circuit_weights)
topk_trig_loss_diffs, indices1 = torch.topk(np.abs(trig_loss_diffs), k)
topk_excluded_loss_diffs, indices2 = torch.topk(np.abs(excluded_loss_diffs), k)
A = weight_ids[indices1]
B = weight_ids[indices2]
intersection = torch.tensor([x for x in A if x in B])
print(len(intersection), k - len(intersection))

number of gen circuit weights:  100075
number of mem circuit weights:  226687
9954 46


In [ ]:
k = 1000

# Get top k elements from both lists
topk_trig_loss_diffs, indices1 = torch.topk(torch.abs(trig_loss_diffs), k)
topk_excluded_loss_diffs, indices2 = torch.topk(torch.abs(excluded_loss_diffs), k)

# Get the corresponding weight ids
A = weight_ids[indices1]
B = weight_ids[indices2]

# Find unique elements in A that are not in B and vice versa
unique_to_A = torch.tensor([x for x in A if x not in B])
unique_to_B = torch.tensor([x for x in B if x not in A])

# Find intersection
intersection = torch.tensor([x for x in A if x in B])

print("Intersection:", len(intersection))
print("Unique to A:", len(unique_to_A))
print("Unique to B:", len(unique_to_B))
print(unique_to_A)
print(unique_to_B)

In [27]:
import copy

def prune_to_obtain_circuit(gen_model, gen_model_indices, mem_model, mem_model_indices, other_model, other_model_indices):
    
    gen_state_dict = gen_model.state_dict()
    mem_state_dict = mem_model.state_dict()
    other_state_dict = other_model.state_dict()
    
    for name, param in gen_model.named_parameters():
        gen_indices = gen_model_indices[name]
        gen_state_dict[name][~gen_indices] = 0.0
        
        mem_indices = mem_model_indices[name]
        mem_state_dict[name][~mem_indices] = 0.0
        
        other_indices = other_model_indices[name]
        other_state_dict[name][~other_indices] = 0.0
        
    gen_model.load_state_dict(gen_state_dict)
    mem_model.load_state_dict(mem_state_dict)
    other_model.load_state_dict(other_state_dict)
            
    return gen_model, mem_model, other_model

gen_models = []
mem_models = []
other_models = []

for model in tqdm(models_saved):
    gen_model = copy.deepcopy(model)
    mem_model = copy.deepcopy(model)
    other_model = copy.deepcopy(model)
    
    gen_model, mem_model, other_model = prune_to_obtain_circuit(gen_model, gen_model_indices, mem_model, mem_model_indices, other_model, other_model_indices)
    
    gen_models.append(gen_model)
    mem_models.append(mem_model)
    other_models.append(other_model)
    
torch.save(gen_models, 'gen_models.pt')
torch.save(mem_models, 'mem_models.pt')
torch.save(other_models, 'other_models.pt')

  0%|          | 0/300 [00:00<?, ?it/s]

In [9]:
gen_models = torch.load('gen_models.pt')
mem_models = torch.load('mem_models.pt')
other_models = torch.load('other_models.pt')

In [ ]:
rlct_estimates_final_gen = obtain_rlct_estimates(train_loader, gen_models, criterion, runs)
torch.save(rlct_estimates_final_gen, 'rlct_estimates_final_gen.pt')

Chain 0:  67%|██████▋   | 333/500 [00:03<00:01, 99.01it/s]

Chain 0:  69%|██████▊   | 343/500 [00:03<00:01, 98.98it/s]

Chain 0:  71%|███████   | 353/500 [00:03<00:01, 99.02it/s]

Chain 0:  73%|███████▎  | 363/500 [00:03<00:01, 99.08it/s]

Chain 0:  75%|███████▍  | 373/500 [00:03<00:01, 99.08it/s]

Chain 0:  77%|███████▋  | 383/500 [00:03<00:01, 99.04it/s]

Chain 0:  79%|███████▊  | 393/500 [00:03<00:01, 98.96it/s]

Chain 0:  81%|████████  | 403/500 [00:04<00:00, 99.04it/s]

Chain 0:  83%|████████▎ | 413/500 [00:04<00:00, 99.05it/s]

Chain 0:  85%|████████▍ | 423/500 [00:04<00:00, 99.08it/s]

Chain 0:  87%|████████▋ | 433/500 [00:04<00:00, 99.01it/s]

Chain 0:  89%|████████▊ | 443/500 [00:04<00:00, 99.09it/s]

Chain 0:  91%|█████████ | 453/500 [00:04<00:00, 99.08it/s]

Chain 0:  93%|█████████▎| 463/500 [00:04<00:00, 99.04it/s]

Chain 0:  95%|█████████▍| 473/500 [00:04<00:00, 99.03it/s]

Chain 0:  97%|█████████▋| 483/500 [00:04<00:00, 99.04it/s]

Chain 0: 100%|██████████| 500/500 [00:05

Chain 0:  73%|███████▎  | 364/500 [00:03<00:01, 99.38it/s]

Chain 0:  75%|███████▍  | 374/500 [00:03<00:01, 99.40it/s]

Chain 0:  77%|███████▋  | 384/500 [00:03<00:01, 99.35it/s]

Chain 0:  79%|███████▉  | 394/500 [00:03<00:01, 99.44it/s]

Chain 0:  81%|████████  | 404/500 [00:04<00:00, 99.41it/s]

Chain 0:  83%|████████▎ | 414/500 [00:04<00:00, 99.36it/s]

Chain 0:  85%|████████▍ | 424/500 [00:04<00:00, 99.37it/s]

Chain 0:  87%|████████▋ | 434/500 [00:04<00:00, 99.35it/s]

Chain 0:  89%|████████▉ | 444/500 [00:04<00:00, 99.34it/s]

Chain 0:  91%|█████████ | 454/500 [00:04<00:00, 99.29it/s]

Chain 0:  93%|█████████▎| 464/500 [00:04<00:00, 99.27it/s]

Chain 0:  95%|█████████▍| 474/500 [00:04<00:00, 99.19it/s]

Chain 0:  97%|█████████▋| 484/500 [00:04<00:00, 99.21it/s]

Chain 0: 100%|██████████| 500/500 [00:04<00:00, 100.14it/s][A


Chain 0:   0%|          | 0/500 [00:00<?, ?it/s]

Chain 0:   2%|▏         | 11/500 [00:00<00:04, 104.49it/s]

Chain 0:   4%|▍         | 22/500 [00:00<00:04,

Chain 0:  79%|███████▉  | 394/500 [00:03<00:01, 99.12it/s]

Chain 0:  81%|████████  | 404/500 [00:04<00:00, 99.09it/s]

Chain 0:  83%|████████▎ | 414/500 [00:04<00:00, 99.14it/s]

Chain 0:  85%|████████▍ | 424/500 [00:04<00:00, 99.16it/s]

Chain 0:  87%|████████▋ | 434/500 [00:04<00:00, 99.19it/s]

Chain 0:  89%|████████▉ | 444/500 [00:04<00:00, 99.18it/s]

Chain 0:  91%|█████████ | 454/500 [00:04<00:00, 99.25it/s]

Chain 0:  93%|█████████▎| 464/500 [00:04<00:00, 99.08it/s]

Chain 0:  95%|█████████▍| 474/500 [00:04<00:00, 99.17it/s]

Chain 0:  97%|█████████▋| 484/500 [00:04<00:00, 99.20it/s]

Chain 0: 100%|██████████| 500/500 [00:04<00:00, 100.07it/s][A


Chain 0:   0%|          | 0/500 [00:00<?, ?it/s]

Chain 0:   2%|▏         | 11/500 [00:00<00:04, 104.46it/s]

Chain 0:   4%|▍         | 22/500 [00:00<00:04, 104.36it/s]

Chain 0:   7%|▋         | 33/500 [00:00<00:04, 104.38it/s]

Chain 0:   9%|▉         | 44/500 [00:00<00:04, 104.27it/s]

Chain 0:  11%|█         | 55/500 [00:00<00:04,

Chain 0:  85%|████████▍ | 424/500 [00:04<00:00, 99.04it/s]

Chain 0:  87%|████████▋ | 434/500 [00:04<00:00, 99.03it/s]

Chain 0:  89%|████████▉ | 444/500 [00:04<00:00, 99.05it/s]

Chain 0:  91%|█████████ | 454/500 [00:04<00:00, 99.08it/s]

Chain 0:  93%|█████████▎| 464/500 [00:04<00:00, 99.12it/s]

Chain 0:  95%|█████████▍| 474/500 [00:04<00:00, 99.03it/s]

Chain 0:  97%|█████████▋| 484/500 [00:04<00:00, 99.07it/s]

Chain 0: 100%|██████████| 500/500 [00:04<00:00, 100.01it/s][A


Chain 0:   0%|          | 0/500 [00:00<?, ?it/s]

Chain 0:   2%|▏         | 11/500 [00:00<00:04, 104.35it/s]

Chain 0:   4%|▍         | 22/500 [00:00<00:04, 104.26it/s]

Chain 0:   7%|▋         | 33/500 [00:00<00:04, 104.20it/s]

Chain 0:   9%|▉         | 44/500 [00:00<00:04, 104.02it/s]

Chain 0:  11%|█         | 55/500 [00:00<00:04, 104.17it/s]

Chain 0:  13%|█▎        | 66/500 [00:00<00:04, 104.18it/s]

Chain 0:  15%|█▌        | 77/500 [00:00<00:04, 104.26it/s]

Chain 0:  18%|█▊        | 88/500 [00:00<00:03,

Chain 0:  91%|█████████ | 454/500 [00:04<00:00, 99.11it/s]

Chain 0:  93%|█████████▎| 464/500 [00:04<00:00, 99.15it/s]

Chain 0:  95%|█████████▍| 474/500 [00:04<00:00, 99.16it/s]

Chain 0:  97%|█████████▋| 484/500 [00:04<00:00, 99.16it/s]

Chain 0: 100%|██████████| 500/500 [00:04<00:00, 100.07it/s][A


Chain 0:   0%|          | 0/500 [00:00<?, ?it/s]

Chain 0:   2%|▏         | 11/500 [00:00<00:04, 104.57it/s]

Chain 0:   4%|▍         | 22/500 [00:00<00:04, 104.36it/s]

Chain 0:   7%|▋         | 33/500 [00:00<00:04, 104.30it/s]

Chain 0:   9%|▉         | 44/500 [00:00<00:04, 104.14it/s]

Chain 0:  11%|█         | 55/500 [00:00<00:04, 104.21it/s]

Chain 0:  13%|█▎        | 66/500 [00:00<00:04, 104.28it/s]

Chain 0:  15%|█▌        | 77/500 [00:00<00:04, 104.25it/s]

Chain 0:  18%|█▊        | 88/500 [00:00<00:03, 104.30it/s]

Chain 0:  20%|█▉        | 99/500 [00:00<00:03, 104.29it/s]

Chain 0:  22%|██▏       | 110/500 [00:01<00:03, 101.34it/s]

Chain 0:  24%|██▍       | 121/500 [00:01<00:0

...

Chain 0:  93%|█████████▎| 464/500 [00:04<00:00, 99.21it/s]

Chain 0:  95%|█████████▍| 474/500 [00:04<00:00, 99.22it/s]

Chain 0:  97%|█████████▋| 484/500 [00:04<00:00, 99.26it/s]

Chain 0: 100%|██████████| 500/500 [00:04<00:00, 100.07it/s][A


Chain 0:   0%|          | 0/500 [00:00<?, ?it/s]

Chain 0:   2%|▏         | 11/500 [00:00<00:04, 104.27it/s]

Chain 0:   4%|▍         | 22/500 [00:00<00:04, 104.23it/s]

Chain 0:   7%|▋         | 33/500 [00:00<00:04, 104.17it/s]

Chain 0:   9%|▉         | 44/500 [00:00<00:04, 104.28it/s]

Chain 0:  11%|█         | 55/500 [00:00<00:04, 104.17it/s]

Chain 0:  13%|█▎        | 66/500 [00:00<00:04, 104.23it/s]

Chain 0:  15%|█▌        | 77/500 [00:00<00:04, 104.35it/s]

Chain 0:  18%|█▊        | 88/500 [00:00<00:03, 104.27it/s]

Chain 0:  20%|█▉        | 99/500 [00:00<00:03, 104.29it/s]

Chain 0:  22%|██▏       | 110/500 [00:01<00:03, 101.42it/s]

Chain 0:  24%|██▍       | 121/500 [00:01<00:03, 100.75it/s]

Chain 0:  26%|██▋       | 132/500 [00:01<00:

Chain 0: 100%|██████████| 500/500 [00:04<00:00, 100.02it/s][A


Chain 0:   0%|          | 0/500 [00:00<?, ?it/s]

Chain 0:   2%|▏         | 11/500 [00:00<00:04, 104.10it/s]

Chain 0:   4%|▍         | 22/500 [00:00<00:04, 104.30it/s]

Chain 0:   7%|▋         | 33/500 [00:00<00:04, 104.26it/s]

Chain 0:   9%|▉         | 44/500 [00:00<00:04, 104.32it/s]

Chain 0:  11%|█         | 55/500 [00:00<00:04, 104.35it/s]

Chain 0:  13%|█▎        | 66/500 [00:00<00:04, 104.38it/s]

Chain 0:  15%|█▌        | 77/500 [00:00<00:04, 104.41it/s]

Chain 0:  18%|█▊        | 88/500 [00:00<00:03, 104.32it/s]

Chain 0:  20%|█▉        | 99/500 [00:00<00:03, 104.31it/s]

Chain 0:  22%|██▏       | 110/500 [00:01<00:03, 101.44it/s]

Chain 0:  24%|██▍       | 121/500 [00:01<00:03, 100.74it/s]

Chain 0:  26%|██▋       | 132/500 [00:01<00:03, 100.27it/s]

Chain 0:  29%|██▊       | 143/500 [00:01<00:03, 100.02it/s]

Chain 0:  31%|███       | 154/500 [00:01<00:03, 99.84it/s] 

Chain 0:  33%|███▎      | 164/500 [00:01<

Chain 0:   4%|▍         | 22/500 [00:00<00:04, 104.54it/s]

Chain 0:   7%|▋         | 33/500 [00:00<00:04, 104.28it/s]

Chain 0:   9%|▉         | 44/500 [00:00<00:04, 104.40it/s]

Chain 0:  11%|█         | 55/500 [00:00<00:04, 104.43it/s]

Chain 0:  13%|█▎        | 66/500 [00:00<00:04, 104.34it/s]

Chain 0:  15%|█▌        | 77/500 [00:00<00:04, 104.34it/s]

Chain 0:  18%|█▊        | 88/500 [00:00<00:03, 104.36it/s]

Chain 0:  20%|█▉        | 99/500 [00:00<00:03, 104.37it/s]

Chain 0:  22%|██▏       | 110/500 [00:01<00:03, 101.40it/s]

Chain 0:  24%|██▍       | 121/500 [00:01<00:03, 100.77it/s]

Chain 0:  26%|██▋       | 132/500 [00:01<00:03, 100.37it/s]

Chain 0:  29%|██▊       | 143/500 [00:01<00:03, 100.04it/s]

Chain 0:  31%|███       | 154/500 [00:01<00:03, 99.82it/s] 

Chain 0:  33%|███▎      | 164/500 [00:01<00:03, 99.62it/s]

Chain 0:  35%|███▍      | 174/500 [00:01<00:03, 99.45it/s]

Chain 0:  37%|███▋      | 184/500 [00:01<00:03, 99.35it/s]

Chain 0:  39%|███▉      | 194/500 [

Chain 0:  11%|█         | 55/500 [00:00<00:04, 104.35it/s]

Chain 0:  13%|█▎        | 66/500 [00:00<00:04, 104.34it/s]

Chain 0:  15%|█▌        | 77/500 [00:00<00:04, 104.32it/s]

Chain 0:  18%|█▊        | 88/500 [00:00<00:03, 104.28it/s]

Chain 0:  20%|█▉        | 99/500 [00:00<00:03, 104.34it/s]

Chain 0:  22%|██▏       | 110/500 [00:01<00:03, 101.44it/s]

Chain 0:  24%|██▍       | 121/500 [00:01<00:03, 100.76it/s]

Chain 0:  26%|██▋       | 132/500 [00:01<00:03, 100.32it/s]

Chain 0:  29%|██▊       | 143/500 [00:01<00:03, 100.00it/s]

Chain 0:  31%|███       | 154/500 [00:01<00:03, 99.82it/s] 

Chain 0:  33%|███▎      | 164/500 [00:01<00:03, 99.61it/s]

Chain 0:  35%|███▍      | 174/500 [00:01<00:03, 99.42it/s]

Chain 0:  37%|███▋      | 184/500 [00:01<00:03, 99.32it/s]

Chain 0:  39%|███▉      | 194/500 [00:01<00:03, 99.26it/s]

Chain 0:  41%|████      | 204/500 [00:02<00:02, 99.17it/s]

Chain 0:  43%|████▎     | 214/500 [00:02<00:02, 99.20it/s]

Chain 0:  45%|████▍     | 224/500 [

Chain 0:  18%|█▊        | 88/500 [00:00<00:03, 104.33it/s]

Chain 0:  20%|█▉        | 99/500 [00:00<00:03, 104.30it/s]

Chain 0:  22%|██▏       | 110/500 [00:01<00:03, 101.37it/s]

Chain 0:  24%|██▍       | 121/500 [00:01<00:03, 100.69it/s]

Chain 0:  26%|██▋       | 132/500 [00:01<00:03, 100.19it/s]

Chain 0:  29%|██▊       | 143/500 [00:01<00:03, 99.91it/s] 

Chain 0:  31%|███       | 153/500 [00:01<00:03, 99.73it/s]

Chain 0:  33%|███▎      | 163/500 [00:01<00:03, 99.52it/s]

Chain 0:  35%|███▍      | 173/500 [00:01<00:03, 99.40it/s]

Chain 0:  37%|███▋      | 183/500 [00:01<00:03, 99.31it/s]

Chain 0:  39%|███▊      | 193/500 [00:01<00:03, 99.13it/s]

Chain 0:  41%|████      | 203/500 [00:02<00:02, 99.03it/s]

Chain 0:  43%|████▎     | 213/500 [00:02<00:02, 99.09it/s]

Chain 0:  45%|████▍     | 223/500 [00:02<00:02, 99.16it/s]

Chain 0:  47%|████▋     | 233/500 [00:02<00:02, 99.16it/s]

Chain 0:  49%|████▊     | 243/500 [00:02<00:02, 99.21it/s]

Chain 0:  51%|█████     | 253/500 [0

In [ ]:
rlct_estimates_final_mem = obtain_rlct_estimates(train_loader, mem_models, criterion, runs)
torch.save(rlct_estimates_final_mem, 'rlct_estimates_final_mem.pt')

Chain 0:   7%|▋         | 33/500 [00:00<00:04, 104.20it/s]

Chain 0:   9%|▉         | 44/500 [00:00<00:04, 104.31it/s]

Chain 0:  11%|█         | 55/500 [00:00<00:04, 104.38it/s]

Chain 0:  13%|█▎        | 66/500 [00:00<00:04, 104.42it/s]

Chain 0:  15%|█▌        | 77/500 [00:00<00:04, 104.45it/s]

Chain 0:  18%|█▊        | 88/500 [00:00<00:03, 104.39it/s]

Chain 0:  20%|█▉        | 99/500 [00:00<00:03, 104.42it/s]

Chain 0:  22%|██▏       | 110/500 [00:01<00:03, 101.51it/s]

Chain 0:  24%|██▍       | 121/500 [00:01<00:03, 100.87it/s]

Chain 0:  26%|██▋       | 132/500 [00:01<00:03, 100.34it/s]

Chain 0:  29%|██▊       | 143/500 [00:01<00:03, 99.98it/s] 

Chain 0:  31%|███       | 154/500 [00:01<00:03, 99.80it/s]

Chain 0:  33%|███▎      | 164/500 [00:01<00:03, 99.58it/s]

Chain 0:  35%|███▍      | 174/500 [00:01<00:03, 99.47it/s]

Chain 0:  37%|███▋      | 184/500 [00:01<00:03, 99.43it/s]

Chain 0:  39%|███▉      | 194/500 [00:01<00:03, 99.43it/s]

Chain 0:  41%|████      | 204/500 [0

Chain 0:  13%|█▎        | 66/500 [00:00<00:04, 104.36it/s]

Chain 0:  15%|█▌        | 77/500 [00:00<00:04, 104.27it/s]

Chain 0:  18%|█▊        | 88/500 [00:00<00:03, 104.28it/s]

Chain 0:  20%|█▉        | 99/500 [00:00<00:03, 104.25it/s]

Chain 0:  22%|██▏       | 110/500 [00:01<00:03, 101.32it/s]

Chain 0:  24%|██▍       | 121/500 [00:01<00:03, 100.71it/s]

Chain 0:  26%|██▋       | 132/500 [00:01<00:03, 100.27it/s]

Chain 0:  29%|██▊       | 143/500 [00:01<00:03, 99.96it/s] 

Chain 0:  31%|███       | 154/500 [00:01<00:03, 99.77it/s]

Chain 0:  33%|███▎      | 164/500 [00:01<00:03, 99.55it/s]

Chain 0:  35%|███▍      | 174/500 [00:01<00:03, 99.47it/s]

Chain 0:  37%|███▋      | 184/500 [00:01<00:03, 99.34it/s]

Chain 0:  39%|███▉      | 194/500 [00:01<00:03, 99.31it/s]

Chain 0:  41%|████      | 204/500 [00:02<00:02, 99.31it/s]

Chain 0:  43%|████▎     | 214/500 [00:02<00:02, 99.29it/s]

Chain 0:  45%|████▍     | 224/500 [00:02<00:02, 99.32it/s]

Chain 0:  47%|████▋     | 234/500 [0

Chain 0:  20%|█▉        | 99/500 [00:00<00:03, 104.35it/s]

Chain 0:  22%|██▏       | 110/500 [00:01<00:03, 101.39it/s]

Chain 0:  24%|██▍       | 121/500 [00:01<00:03, 100.62it/s]

Chain 0:  26%|██▋       | 132/500 [00:01<00:03, 100.19it/s]

Chain 0:  29%|██▊       | 143/500 [00:01<00:03, 99.83it/s] 

Chain 0:  31%|███       | 153/500 [00:01<00:03, 99.60it/s]

Chain 0:  33%|███▎      | 163/500 [00:01<00:03, 99.41it/s]

Chain 0:  35%|███▍      | 173/500 [00:01<00:03, 99.34it/s]

Chain 0:  37%|███▋      | 183/500 [00:01<00:03, 99.24it/s]

Chain 0:  39%|███▊      | 193/500 [00:01<00:03, 99.21it/s]

Chain 0:  41%|████      | 203/500 [00:02<00:02, 99.19it/s]

Chain 0:  43%|████▎     | 213/500 [00:02<00:02, 99.16it/s]

Chain 0:  45%|████▍     | 223/500 [00:02<00:02, 99.19it/s]

Chain 0:  47%|████▋     | 233/500 [00:02<00:02, 99.20it/s]

Chain 0:  49%|████▊     | 243/500 [00:02<00:02, 99.12it/s]

Chain 0:  51%|█████     | 253/500 [00:02<00:02, 99.06it/s]

Chain 0:  53%|█████▎    | 263/500 [0

Chain 0:  26%|██▋       | 132/500 [00:01<00:03, 100.24it/s]

Chain 0:  29%|██▊       | 143/500 [00:01<00:03, 99.95it/s] 

Chain 0:  31%|███       | 154/500 [00:01<00:03, 99.75it/s]

Chain 0:  33%|███▎      | 164/500 [00:01<00:03, 99.55it/s]

Chain 0:  35%|███▍      | 174/500 [00:01<00:03, 99.42it/s]

Chain 0:  37%|███▋      | 184/500 [00:01<00:03, 99.34it/s]

Chain 0:  39%|███▉      | 194/500 [00:01<00:03, 99.34it/s]

Chain 0:  41%|████      | 204/500 [00:02<00:02, 99.32it/s]

Chain 0:  43%|████▎     | 214/500 [00:02<00:02, 99.27it/s]

Chain 0:  45%|████▍     | 224/500 [00:02<00:02, 99.26it/s]

Chain 0:  47%|████▋     | 234/500 [00:02<00:02, 99.30it/s]

Chain 0:  49%|████▉     | 244/500 [00:02<00:02, 99.31it/s]

Chain 0:  51%|█████     | 254/500 [00:02<00:02, 99.11it/s]

Chain 0:  53%|█████▎    | 264/500 [00:02<00:02, 99.18it/s]

Chain 0:  55%|█████▍    | 274/500 [00:02<00:02, 99.20it/s]

Chain 0:  57%|█████▋    | 284/500 [00:02<00:02, 99.15it/s]

Chain 0:  59%|█████▉    | 294/500 [00:

Chain 0:  33%|███▎      | 164/500 [00:01<00:03, 99.64it/s]

Chain 0:  35%|███▍      | 174/500 [00:01<00:03, 99.50it/s]

Chain 0:  37%|███▋      | 184/500 [00:01<00:03, 99.39it/s]

Chain 0:  39%|███▉      | 194/500 [00:01<00:03, 99.37it/s]

Chain 0:  41%|████      | 204/500 [00:02<00:02, 99.34it/s]

Chain 0:  43%|████▎     | 214/500 [00:02<00:02, 99.29it/s]

Chain 0:  45%|████▍     | 224/500 [00:02<00:02, 99.20it/s]

Chain 0:  47%|████▋     | 234/500 [00:02<00:02, 99.25it/s]

Chain 0:  49%|████▉     | 244/500 [00:02<00:02, 99.22it/s]

Chain 0:  51%|█████     | 254/500 [00:02<00:02, 99.22it/s]

Chain 0:  53%|█████▎    | 264/500 [00:02<00:02, 99.25it/s]

Chain 0:  55%|█████▍    | 274/500 [00:02<00:02, 99.23it/s]

Chain 0:  57%|█████▋    | 284/500 [00:02<00:02, 99.26it/s]

Chain 0:  59%|█████▉    | 294/500 [00:02<00:02, 99.23it/s]

Chain 0:  61%|██████    | 304/500 [00:03<00:01, 99.26it/s]

Chain 0:  63%|██████▎   | 314/500 [00:03<00:01, 99.30it/s]

Chain 0:  65%|██████▍   | 324/500 [00:03

...

Chain 0:  37%|███▋      | 184/500 [00:01<00:03, 99.38it/s]

Chain 0:  39%|███▉      | 194/500 [00:01<00:03, 99.41it/s]

Chain 0:  41%|████      | 204/500 [00:02<00:02, 99.34it/s]

Chain 0:  43%|████▎     | 214/500 [00:02<00:02, 99.20it/s]

Chain 0:  45%|████▍     | 224/500 [00:02<00:02, 99.19it/s]

Chain 0:  47%|████▋     | 234/500 [00:02<00:02, 99.11it/s]

Chain 0:  49%|████▉     | 244/500 [00:02<00:02, 99.07it/s]

Chain 0:  51%|█████     | 254/500 [00:02<00:02, 99.08it/s]

Chain 0:  53%|█████▎    | 264/500 [00:02<00:02, 99.20it/s]

Chain 0:  55%|█████▍    | 274/500 [00:02<00:02, 99.08it/s]

Chain 0:  57%|█████▋    | 284/500 [00:02<00:02, 99.15it/s]

Chain 0:  59%|█████▉    | 294/500 [00:02<00:02, 99.04it/s]

Chain 0:  61%|██████    | 304/500 [00:03<00:01, 99.09it/s]

Chain 0:  63%|██████▎   | 314/500 [00:03<00:01, 98.83it/s]

Chain 0:  65%|██████▍   | 324/500 [00:03<00:01, 98.89it/s]

Chain 0:  67%|██████▋   | 334/500 [00:03<00:01, 99.01it/s]

Chain 0:  69%|██████▉   | 344/500 [00:03

Chain 0:  43%|████▎     | 214/500 [00:02<00:02, 99.37it/s]

Chain 0:  45%|████▍     | 224/500 [00:02<00:02, 99.27it/s]

Chain 0:  47%|████▋     | 234/500 [00:02<00:02, 99.22it/s]

Chain 0:  49%|████▉     | 244/500 [00:02<00:02, 99.23it/s]

Chain 0:  51%|█████     | 254/500 [00:02<00:02, 99.28it/s]

Chain 0:  53%|█████▎    | 264/500 [00:02<00:02, 99.34it/s]

Chain 0:  55%|█████▍    | 274/500 [00:02<00:02, 99.36it/s]

Chain 0:  57%|█████▋    | 284/500 [00:02<00:02, 99.38it/s]

Chain 0:  59%|█████▉    | 294/500 [00:02<00:02, 99.40it/s]

Chain 0:  61%|██████    | 304/500 [00:03<00:01, 99.40it/s]

Chain 0:  63%|██████▎   | 314/500 [00:03<00:01, 99.30it/s]

Chain 0:  65%|██████▍   | 324/500 [00:03<00:01, 99.30it/s]

Chain 0:  67%|██████▋   | 334/500 [00:03<00:01, 99.28it/s]

Chain 0:  69%|██████▉   | 344/500 [00:03<00:01, 99.29it/s]

Chain 0:  71%|███████   | 354/500 [00:03<00:01, 99.24it/s]

Chain 0:  73%|███████▎  | 364/500 [00:03<00:01, 99.31it/s]

Chain 0:  75%|███████▍  | 374/500 [00:03

Chain 0:  49%|████▉     | 244/500 [00:02<00:02, 99.23it/s]

Chain 0:  51%|█████     | 254/500 [00:02<00:02, 99.25it/s]

Chain 0:  53%|█████▎    | 264/500 [00:02<00:02, 99.09it/s]

Chain 0:  55%|█████▍    | 274/500 [00:02<00:02, 99.01it/s]

Chain 0:  57%|█████▋    | 284/500 [00:02<00:02, 99.14it/s]

Chain 0:  59%|█████▉    | 294/500 [00:02<00:02, 99.13it/s]

Chain 0:  61%|██████    | 304/500 [00:03<00:01, 99.24it/s]

Chain 0:  63%|██████▎   | 314/500 [00:03<00:01, 99.26it/s]

Chain 0:  65%|██████▍   | 324/500 [00:03<00:01, 99.24it/s]

Chain 0:  67%|██████▋   | 334/500 [00:03<00:01, 99.30it/s]

Chain 0:  69%|██████▉   | 344/500 [00:03<00:01, 99.37it/s]

Chain 0:  71%|███████   | 354/500 [00:03<00:01, 99.38it/s]

Chain 0:  73%|███████▎  | 364/500 [00:03<00:01, 99.35it/s]

Chain 0:  75%|███████▍  | 374/500 [00:03<00:01, 99.41it/s]

Chain 0:  77%|███████▋  | 384/500 [00:03<00:01, 99.43it/s]

Chain 0:  79%|███████▉  | 394/500 [00:03<00:01, 99.38it/s]

Chain 0:  81%|████████  | 404/500 [00:04

Chain 0:  55%|█████▍    | 274/500 [00:02<00:02, 99.20it/s]

Chain 0:  57%|█████▋    | 284/500 [00:02<00:02, 99.23it/s]

Chain 0:  59%|█████▉    | 294/500 [00:02<00:02, 99.23it/s]

Chain 0:  61%|██████    | 304/500 [00:03<00:01, 99.30it/s]

Chain 0:  63%|██████▎   | 314/500 [00:03<00:01, 99.32it/s]

Chain 0:  65%|██████▍   | 324/500 [00:03<00:01, 99.31it/s]

Chain 0:  67%|██████▋   | 334/500 [00:03<00:01, 99.30it/s]

Chain 0:  69%|██████▉   | 344/500 [00:03<00:01, 99.21it/s]

Chain 0:  71%|███████   | 354/500 [00:03<00:01, 99.28it/s]

Chain 0:  73%|███████▎  | 364/500 [00:03<00:01, 99.30it/s]

Chain 0:  75%|███████▍  | 374/500 [00:03<00:01, 99.26it/s]

Chain 0:  77%|███████▋  | 384/500 [00:03<00:01, 99.22it/s]

Chain 0:  79%|███████▉  | 394/500 [00:03<00:01, 99.27it/s]

Chain 0:  81%|████████  | 404/500 [00:04<00:00, 99.25it/s]

Chain 0:  83%|████████▎ | 414/500 [00:04<00:00, 99.20it/s]

Chain 0:  85%|████████▍ | 424/500 [00:04<00:00, 99.18it/s]

Chain 0:  87%|████████▋ | 434/500 [00:04

Chain 0:  61%|██████    | 304/500 [00:03<00:01, 99.34it/s]

Chain 0:  63%|██████▎   | 314/500 [00:03<00:01, 99.39it/s]

Chain 0:  65%|██████▍   | 324/500 [00:03<00:01, 99.37it/s]

Chain 0:  67%|██████▋   | 334/500 [00:03<00:01, 99.38it/s]

Chain 0:  69%|██████▉   | 344/500 [00:03<00:01, 99.35it/s]

Chain 0:  71%|███████   | 354/500 [00:03<00:01, 99.31it/s]

Chain 0:  73%|███████▎  | 364/500 [00:03<00:01, 99.25it/s]

Chain 0:  75%|███████▍  | 374/500 [00:03<00:01, 99.28it/s]

Chain 0:  77%|███████▋  | 384/500 [00:03<00:01, 99.31it/s]

Chain 0:  79%|███████▉  | 394/500 [00:03<00:01, 99.28it/s]

Chain 0:  81%|████████  | 404/500 [00:04<00:00, 99.29it/s]

Chain 0:  83%|████████▎ | 414/500 [00:04<00:00, 99.30it/s]

Chain 0:  85%|████████▍ | 424/500 [00:04<00:00, 99.35it/s]

Chain 0:  87%|████████▋ | 434/500 [00:04<00:00, 99.30it/s]

Chain 0:  89%|████████▉ | 444/500 [00:04<00:00, 99.26it/s]

Chain 0:  91%|█████████ | 454/500 [00:04<00:00, 99.31it/s]

Chain 0:  93%|█████████▎| 464/500 [00:04

In [ ]:
rlct_estimates_final_other = obtain_rlct_estimates(train_loader, other_models, criterion, runs)
torch.save(rlct_estimates_final_other, 'rlct_estimates_final_other.pt')

Chain 0:   7%|▋         | 33/500 [00:00<00:04, 104.53it/s]

Chain 0:   9%|▉         | 44/500 [00:00<00:04, 104.52it/s]

Chain 0:  11%|█         | 55/500 [00:00<00:04, 104.47it/s]

Chain 0:  13%|█▎        | 66/500 [00:00<00:04, 104.49it/s]

Chain 0:  15%|█▌        | 77/500 [00:00<00:04, 104.37it/s]

Chain 0:  18%|█▊        | 88/500 [00:00<00:03, 104.35it/s]

Chain 0:  20%|█▉        | 99/500 [00:00<00:03, 104.38it/s]

Chain 0:  22%|██▏       | 110/500 [00:01<00:03, 101.46it/s]

Chain 0:  24%|██▍       | 121/500 [00:01<00:03, 100.79it/s]

Chain 0:  26%|██▋       | 132/500 [00:01<00:03, 100.39it/s]

Chain 0:  29%|██▊       | 143/500 [00:01<00:03, 100.01it/s]

Chain 0:  31%|███       | 154/500 [00:01<00:03, 99.75it/s] 

Chain 0:  33%|███▎      | 164/500 [00:01<00:03, 99.63it/s]

Chain 0:  35%|███▍      | 174/500 [00:01<00:03, 99.54it/s]

Chain 0:  37%|███▋      | 184/500 [00:01<00:03, 99.47it/s]

Chain 0:  39%|███▉      | 194/500 [00:01<00:03, 99.41it/s]

Chain 0:  41%|████      | 204/500 [

Chain 0:  13%|█▎        | 66/500 [00:00<00:04, 104.27it/s]

Chain 0:  15%|█▌        | 77/500 [00:00<00:04, 104.32it/s]

Chain 0:  18%|█▊        | 88/500 [00:00<00:03, 104.36it/s]

Chain 0:  20%|█▉        | 99/500 [00:00<00:03, 104.42it/s]

Chain 0:  22%|██▏       | 110/500 [00:01<00:03, 101.47it/s]

Chain 0:  24%|██▍       | 121/500 [00:01<00:03, 100.87it/s]

Chain 0:  26%|██▋       | 132/500 [00:01<00:03, 100.41it/s]

Chain 0:  29%|██▊       | 143/500 [00:01<00:03, 100.05it/s]

Chain 0:  31%|███       | 154/500 [00:01<00:03, 99.85it/s] 

Chain 0:  33%|███▎      | 164/500 [00:01<00:03, 99.71it/s]

Chain 0:  35%|███▍      | 174/500 [00:01<00:03, 99.51it/s]

Chain 0:  37%|███▋      | 184/500 [00:01<00:03, 99.49it/s]

Chain 0:  39%|███▉      | 194/500 [00:01<00:03, 99.43it/s]

Chain 0:  41%|████      | 204/500 [00:02<00:02, 99.32it/s]

Chain 0:  43%|████▎     | 214/500 [00:02<00:02, 99.36it/s]

Chain 0:  45%|████▍     | 224/500 [00:02<00:02, 99.24it/s]

Chain 0:  47%|████▋     | 234/500 [

Chain 0:  20%|█▉        | 99/500 [00:00<00:03, 104.32it/s]

Chain 0:  22%|██▏       | 110/500 [00:01<00:03, 101.32it/s]

Chain 0:  24%|██▍       | 121/500 [00:01<00:03, 100.75it/s]

Chain 0:  26%|██▋       | 132/500 [00:01<00:03, 100.36it/s]

Chain 0:  29%|██▊       | 143/500 [00:01<00:03, 100.06it/s]

Chain 0:  31%|███       | 154/500 [00:01<00:03, 99.80it/s] 

Chain 0:  33%|███▎      | 164/500 [00:01<00:03, 99.60it/s]

Chain 0:  35%|███▍      | 174/500 [00:01<00:03, 99.51it/s]

Chain 0:  37%|███▋      | 184/500 [00:01<00:03, 99.41it/s]

Chain 0:  39%|███▉      | 194/500 [00:01<00:03, 99.35it/s]

Chain 0:  41%|████      | 204/500 [00:02<00:02, 99.38it/s]

Chain 0:  43%|████▎     | 214/500 [00:02<00:02, 99.31it/s]

Chain 0:  45%|████▍     | 224/500 [00:02<00:02, 99.31it/s]

Chain 0:  47%|████▋     | 234/500 [00:02<00:02, 99.36it/s]

Chain 0:  49%|████▉     | 244/500 [00:02<00:02, 99.34it/s]

Chain 0:  51%|█████     | 254/500 [00:02<00:02, 99.31it/s]

Chain 0:  53%|█████▎    | 264/500 [

Chain 0:  26%|██▋       | 132/500 [00:01<00:03, 100.31it/s]

Chain 0:  29%|██▊       | 143/500 [00:01<00:03, 100.09it/s]

Chain 0:  31%|███       | 154/500 [00:01<00:03, 99.90it/s] 

Chain 0:  33%|███▎      | 164/500 [00:01<00:03, 99.77it/s]

Chain 0:  35%|███▍      | 174/500 [00:01<00:03, 99.68it/s]

Chain 0:  37%|███▋      | 184/500 [00:01<00:03, 99.59it/s]

Chain 0:  39%|███▉      | 194/500 [00:01<00:03, 99.51it/s]

Chain 0:  41%|████      | 204/500 [00:02<00:02, 99.48it/s]

Chain 0:  43%|████▎     | 214/500 [00:02<00:02, 99.37it/s]

Chain 0:  45%|████▍     | 224/500 [00:02<00:02, 99.25it/s]

Chain 0:  47%|████▋     | 234/500 [00:02<00:02, 99.17it/s]

Chain 0:  49%|████▉     | 244/500 [00:02<00:02, 99.13it/s]

Chain 0:  51%|█████     | 254/500 [00:02<00:02, 99.15it/s]

Chain 0:  53%|█████▎    | 264/500 [00:02<00:02, 99.25it/s]

Chain 0:  55%|█████▍    | 274/500 [00:02<00:02, 99.28it/s]

Chain 0:  57%|█████▋    | 284/500 [00:02<00:02, 99.24it/s]

Chain 0:  59%|█████▉    | 294/500 [00

Chain 0:  33%|███▎      | 164/500 [00:01<00:03, 99.72it/s]

Chain 0:  35%|███▍      | 174/500 [00:01<00:03, 99.62it/s]

Chain 0:  37%|███▋      | 184/500 [00:01<00:03, 99.56it/s]

Chain 0:  39%|███▉      | 194/500 [00:01<00:03, 99.53it/s]

Chain 0:  41%|████      | 204/500 [00:02<00:02, 99.50it/s]

Chain 0:  43%|████▎     | 214/500 [00:02<00:02, 99.44it/s]

Chain 0:  45%|████▍     | 224/500 [00:02<00:02, 99.43it/s]

Chain 0:  47%|████▋     | 234/500 [00:02<00:02, 99.39it/s]

Chain 0:  49%|████▉     | 244/500 [00:02<00:02, 99.35it/s]

Chain 0:  51%|█████     | 254/500 [00:02<00:02, 99.36it/s]

Chain 0:  53%|█████▎    | 264/500 [00:02<00:02, 99.39it/s]

Chain 0:  55%|█████▍    | 274/500 [00:02<00:02, 99.39it/s]

Chain 0:  57%|█████▋    | 284/500 [00:02<00:02, 99.35it/s]

Chain 0:  59%|█████▉    | 294/500 [00:02<00:02, 99.24it/s]

Chain 0:  61%|██████    | 304/500 [00:03<00:01, 99.24it/s]

Chain 0:  63%|██████▎   | 314/500 [00:03<00:01, 99.17it/s]

Chain 0:  65%|██████▍   | 324/500 [00:03

...

Chain 0:  37%|███▋      | 184/500 [00:01<00:03, 99.48it/s]

Chain 0:  39%|███▉      | 194/500 [00:01<00:03, 99.36it/s]

Chain 0:  41%|████      | 204/500 [00:02<00:02, 99.28it/s]

Chain 0:  43%|████▎     | 214/500 [00:02<00:02, 99.24it/s]

Chain 0:  45%|████▍     | 224/500 [00:02<00:02, 99.28it/s]

Chain 0:  47%|████▋     | 234/500 [00:02<00:02, 99.32it/s]

Chain 0:  49%|████▉     | 244/500 [00:02<00:02, 99.20it/s]

Chain 0:  51%|█████     | 254/500 [00:02<00:02, 99.20it/s]

Chain 0:  53%|█████▎    | 264/500 [00:02<00:02, 99.23it/s]

Chain 0:  55%|█████▍    | 274/500 [00:02<00:02, 99.09it/s]

Chain 0:  57%|█████▋    | 284/500 [00:02<00:02, 99.15it/s]

Chain 0:  59%|█████▉    | 294/500 [00:02<00:02, 99.08it/s]

Chain 0:  61%|██████    | 304/500 [00:03<00:01, 99.17it/s]

Chain 0:  63%|██████▎   | 314/500 [00:03<00:01, 99.22it/s]

Chain 0:  65%|██████▍   | 324/500 [00:03<00:01, 99.25it/s]

Chain 0:  67%|██████▋   | 334/500 [00:03<00:01, 99.29it/s]

Chain 0:  69%|██████▉   | 344/500 [00:03

Chain 0:  43%|████▎     | 214/500 [00:02<00:02, 99.24it/s]

Chain 0:  45%|████▍     | 224/500 [00:02<00:02, 99.27it/s]

Chain 0:  47%|████▋     | 234/500 [00:02<00:02, 99.26it/s]

Chain 0:  49%|████▉     | 244/500 [00:02<00:02, 99.29it/s]

Chain 0:  51%|█████     | 254/500 [00:02<00:02, 99.30it/s]

Chain 0:  53%|█████▎    | 264/500 [00:02<00:02, 99.32it/s]

Chain 0:  55%|█████▍    | 274/500 [00:02<00:02, 99.29it/s]

Chain 0:  57%|█████▋    | 284/500 [00:02<00:02, 99.25it/s]

Chain 0:  59%|█████▉    | 294/500 [00:02<00:02, 99.16it/s]

Chain 0:  61%|██████    | 304/500 [00:03<00:01, 99.10it/s]

Chain 0:  63%|██████▎   | 314/500 [00:03<00:01, 99.16it/s]

Chain 0:  65%|██████▍   | 324/500 [00:03<00:01, 99.10it/s]

Chain 0:  67%|██████▋   | 334/500 [00:03<00:01, 99.20it/s]

Chain 0:  69%|██████▉   | 344/500 [00:03<00:01, 99.18it/s]

Chain 0:  71%|███████   | 354/500 [00:03<00:01, 99.24it/s]

Chain 0:  73%|███████▎  | 364/500 [00:03<00:01, 99.21it/s]

Chain 0:  75%|███████▍  | 374/500 [00:03

Chain 0:  49%|████▉     | 244/500 [00:02<00:02, 99.20it/s]

Chain 0:  51%|█████     | 254/500 [00:02<00:02, 99.16it/s]

Chain 0:  53%|█████▎    | 264/500 [00:02<00:02, 99.11it/s]

Chain 0:  55%|█████▍    | 274/500 [00:02<00:02, 99.13it/s]

Chain 0:  57%|█████▋    | 284/500 [00:02<00:02, 99.18it/s]

Chain 0:  59%|█████▉    | 294/500 [00:02<00:02, 99.22it/s]

Chain 0:  61%|██████    | 304/500 [00:03<00:01, 99.21it/s]

Chain 0:  63%|██████▎   | 314/500 [00:03<00:01, 99.18it/s]

Chain 0:  65%|██████▍   | 324/500 [00:03<00:01, 99.19it/s]

Chain 0:  67%|██████▋   | 334/500 [00:03<00:01, 99.09it/s]

Chain 0:  69%|██████▉   | 344/500 [00:03<00:01, 99.19it/s]

Chain 0:  71%|███████   | 354/500 [00:03<00:01, 99.25it/s]

Chain 0:  73%|███████▎  | 364/500 [00:03<00:01, 99.21it/s]

Chain 0:  75%|███████▍  | 374/500 [00:03<00:01, 99.18it/s]

Chain 0:  77%|███████▋  | 384/500 [00:03<00:01, 99.16it/s]

Chain 0:  79%|███████▉  | 394/500 [00:03<00:01, 99.20it/s]

Chain 0:  81%|████████  | 404/500 [00:04

Chain 0:  55%|█████▍    | 274/500 [00:02<00:02, 99.03it/s]

Chain 0:  57%|█████▋    | 284/500 [00:02<00:02, 99.04it/s]

Chain 0:  59%|█████▉    | 294/500 [00:02<00:02, 99.02it/s]

Chain 0:  61%|██████    | 304/500 [00:03<00:01, 98.98it/s]

Chain 0:  63%|██████▎   | 314/500 [00:03<00:01, 99.09it/s]

Chain 0:  65%|██████▍   | 324/500 [00:03<00:01, 99.14it/s]

Chain 0:  67%|██████▋   | 334/500 [00:03<00:01, 99.02it/s]

Chain 0:  69%|██████▉   | 344/500 [00:03<00:01, 99.12it/s]

Chain 0:  71%|███████   | 354/500 [00:03<00:01, 99.19it/s]

Chain 0:  73%|███████▎  | 364/500 [00:03<00:01, 99.15it/s]

Chain 0:  75%|███████▍  | 374/500 [00:03<00:01, 99.12it/s]

Chain 0:  77%|███████▋  | 384/500 [00:03<00:01, 99.11it/s]

Chain 0:  79%|███████▉  | 394/500 [00:03<00:01, 99.11it/s]

Chain 0:  81%|████████  | 404/500 [00:04<00:00, 99.04it/s]

Chain 0:  83%|████████▎ | 414/500 [00:04<00:00, 99.05it/s]

Chain 0:  85%|████████▍ | 424/500 [00:04<00:00, 99.16it/s]

Chain 0:  87%|████████▋ | 434/500 [00:04

Chain 0:  61%|██████    | 304/500 [00:03<00:01, 99.31it/s]

Chain 0:  63%|██████▎   | 314/500 [00:03<00:01, 99.25it/s]

Chain 0:  65%|██████▍   | 324/500 [00:03<00:01, 99.27it/s]

Chain 0:  67%|██████▋   | 334/500 [00:03<00:01, 99.24it/s]

Chain 0:  69%|██████▉   | 344/500 [00:03<00:01, 99.26it/s]

Chain 0:  71%|███████   | 354/500 [00:03<00:01, 99.32it/s]

Chain 0:  73%|███████▎  | 364/500 [00:03<00:01, 99.36it/s]

Chain 0:  75%|███████▍  | 374/500 [00:03<00:01, 99.37it/s]

Chain 0:  77%|███████▋  | 384/500 [00:03<00:01, 99.33it/s]

Chain 0:  79%|███████▉  | 394/500 [00:03<00:01, 99.30it/s]

Chain 0:  81%|████████  | 404/500 [00:04<00:00, 99.33it/s]

Chain 0:  83%|████████▎ | 414/500 [00:04<00:00, 99.33it/s]

Chain 0:  85%|████████▍ | 424/500 [00:04<00:00, 99.41it/s]

Chain 0:  87%|████████▋ | 434/500 [00:04<00:00, 99.42it/s]

Chain 0:  89%|████████▉ | 444/500 [00:04<00:00, 99.40it/s]

Chain 0:  91%|█████████ | 454/500 [00:04<00:00, 99.36it/s]

Chain 0:  93%|█████████▎| 464/500 [00:04

In [28]:
rlct_estimates_final_gen = torch.load('rlct_estimates_final_gen.pt')
rlct_estimates_final_mem = torch.load('rlct_estimates_final_mem.pt')
rlct_estimates_final_other = torch.load('rlct_estimates_final_other.pt')

In [31]:
plot_rlcts_circuits(rlct_estimates_final_gen, rlct_estimates_final_mem, rlct_estimates_final_other)
summed_curves = {}
summed_curves['sgld'] = rlct_estimates_final_gen['sgld'] + rlct_estimates_final_mem['sgld'] + rlct_estimates_final_other['sgld']
plot_rlcts(rlct_estimates_final, dataset='comparison_with_summed_curves', rlct_estimates_final_other=summed_curves)
plot_rlcts(rlct_estimates_final_gen, dataset='gen')
plot_rlcts(rlct_estimates_final_mem, dataset='mem')
plot_rlcts(rlct_estimates_final_other, dataset='other')

/scratch-local/bshaffrey.7546692/ipykernel_2629260/3398407666.py:102: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/scratch-local/bshaffrey.7546692/ipykernel_2629260/3398407666.py:80: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [31]:
print(rlct_estimates_final_gen['sgld'])
print(rlct_estimates_final_mem['sgld'])
print(rlct_estimates_final_other['sgld'])

tensor([ 2.1777,  3.5818,  5.7040,  5.8062,  6.0300,  6.2321,  6.4422,  6.6161,
         6.7699,  6.9180,  7.0782,  7.2368,  7.3487,  7.3906,  7.3560,  7.2135,
         6.9108,  6.5195,  6.0920,  5.6251,  5.0985,  4.5680,  4.0125,  3.5158,
         3.0609,  2.6125,  2.1882,  1.8010,  1.4668,  1.1971,  0.9902,  0.8019,
         0.6111,  0.4557,  0.3091,  0.1980,  0.1336,  0.0958,  0.0853,  0.0757,
         0.0113,  0.0177,  0.0805,  0.1658,  0.2262,  0.2874,  0.3075,  0.2641,
         0.2238,  0.1691,  0.1691,  0.1908,  0.2778,  0.3510,  0.4146,  0.4476,
         0.4605,  0.4919,  0.4766,  0.4919,  0.5507,  0.6167,  0.6368,  0.6167,
         0.5338,  0.3736,  0.1844, -0.0314, -0.2512, -0.4798, -0.7109, -0.9460,
        -1.1738, -1.3533, -1.4999, -1.6126, -1.6915, -1.7390, -1.7551, -1.7470,
        -1.7245, -1.6947, -1.6657, -1.6005, -1.5272, -1.4652, -1.4169, -1.3751,
        -1.3260, -1.2793, -1.2366, -1.1980, -1.1641, -1.1295, -1.0933, -1.0844,
        -1.0651, -1.0418, -1.0192, -0.99

In [13]:
print(all_data.shape)

for name, param in models_saved[-1].named_parameters():
    print(name)
    print(param.shape)

torch.Size([12769, 3])
embed.W_E
torch.Size([128, 114])
pos_embed.W_pos
torch.Size([3, 128])
blocks.0.attn.W_K
torch.Size([4, 32, 128])
blocks.0.attn.W_Q
torch.Size([4, 32, 128])
blocks.0.attn.W_V
torch.Size([4, 32, 128])
blocks.0.attn.W_O
torch.Size([128, 128])
blocks.0.mlp.W_in
torch.Size([512, 128])
blocks.0.mlp.b_in
torch.Size([512])
blocks.0.mlp.W_out
torch.Size([128, 512])
blocks.0.mlp.b_out
torch.Size([128])
unembed.W_U
torch.Size([128, 114])
